In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import IPython.display
import dataclasses
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors
import scipy.stats
import astropy.units as u
import astropy.visualization
import named_arrays as na
import optika
import iris
import ctis
import esis

In [ ]:
%%time
scene = esis.flights.f1.data.synth.scene_iris(
    time_start="2014-07-04 11:40",
    velocity_max=150 * u.km / u.s,
)

In [ ]:
scene.inputs.position = scene.inputs.position - scene.inputs.position.mean()

In [ ]:
scene = scene[{scene.axis_time: 0}]
scene.timedelta = scene.timedelta[{scene.axis_time: 0}]

In [ ]:
scene.show();

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    ax_twin = ax.twiny()
    na.plt.stairs(
        scene.inputs.wavelength,
        scene.outputs.mean((scene.axis_detector_x, scene.axis_detector_y)),
        ax=ax,
        label="original",
        axis=scene.axis_wavelength,
    )
    na.plt.stairs(
        scene.inputs.velocity,
        scene.outputs.mean((scene.axis_detector_x, scene.axis_detector_y)),
        ax=ax_twin,
        label="original",
        axis=scene.axis_wavelength,
        linestyle="none",
    )
    ax.set_xlabel(f"wavelength ({ax.get_xlabel()})")
    ax.set_ylabel(f"average radiance ({ax.get_ylabel()})");

In [ ]:
wavelength_rest = scene.inputs.wavelength_rest

In [ ]:
AA = dict(unit=u.AA, equivalencies=u.doppler_optical(wavelength_rest))

In [ ]:
coordinates_scene = scene.inputs

In [ ]:
position_sensor = na.Cartesian2dVectorArray(
    x=na.arange(0, 300 + 1, axis="sensor_x") * u.pix,
    y=na.arange(0, 300 + 1, axis="sensor_y") * u.pix,
)

In [ ]:
coordinates_sensor = na.SpectralPositionalVectorArray(
    wavelength=scene.inputs.wavelength,
    position=position_sensor,
)

In [ ]:
axis_channel = "channel"

In [ ]:
angle = na.linspace(0, 180, num=4, axis="channel", endpoint=False) * u.deg
angle = angle + 0.4 * u.rad
angle.ndarray

In [ ]:
channel = "dispersion angle = " + angle.to_string_array("%03d")

In [ ]:
ccd = optika.sensors.materials.e2v_ccd97()
absorbance = ccd.absorbance(
    wavelength=wavelength_rest,
    normal=na.Cartesian3dVectorArray(0, 0, -1),
).average
area_effective = 4 * u.mm ** 2 / 0.39 * absorbance.ndarray
area_effective

In [ ]:
instrument = ctis.instruments.IdealInstrument(
    area_effective=area_effective,
    timedelta_exposure=10 * u.s,
    plate_scale=0.77 * u.arcsec / u.pix,
    dispersion=((17.5 * u.km / u.s).to(**AA) - wavelength_rest) / u.pix,
    angle=angle,
    wavelength_ref=wavelength_rest,
    position_ref=na.Cartesian2dVectorArray(150, 150) * u.pix,
    coordinates_scene=coordinates_scene,
    coordinates_sensor=coordinates_sensor,
    channel=channel,
    axis_channel=axis_channel,
    axis_wavelength=scene.axis_wavelength,
    axis_scene_xy=(scene.axis_detector_x, scene.axis_detector_y),
    axis_sensor_xy=("sensor_x", "sensor_y"),
)

In [ ]:
%%time
images = instrument.image(scene)

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = na.plt.subplots(
        axis_rows="rows",
        axis_cols="cols",
        nrows=2,
        ncols=2,
        constrained_layout=True,
        figsize=(10,9),
        sharex=True,
        sharey=True,
        origin="upper",
    )
    ax = axs.combine_axes(("rows", "cols"), axis_channel)
    norm = matplotlib.colors.PowerNorm(
        gamma=0.5,
        vmin=0,
        vmax=images.outputs.value.percentile(99.9).ndarray,
    )
    colorizer = plt.Colorizer(
        cmap="gray",
        norm=norm,
    )
    img = na.plt.pcolormesh(
        # channel,
        images.inputs.position.x,
        images.inputs.position.y,
        C=images.outputs.value,
        ax=ax,
        colorizer=colorizer,
    )
    plt.colorbar(
        mappable=plt.cm.ScalarMappable(colorizer=colorizer),
        ax=ax.ndarray,
        label=f"signal ({images.outputs.unit:latex_inline})",
    )
    na.plt.set_title(channel, ax=ax)
    na.plt.set_aspect("equal", ax=ax)
    na.plt.set_xlabel(f"sensor $x$ ({images.inputs.position.x.unit})", ax=axs[dict(rows=~0)])
    na.plt.set_ylabel(f"sensor $y$ ({images.inputs.position.y.unit})", ax=axs[dict(cols=0)])

In [ ]:
%%time
image_separated = instrument.image(scene, integrate=False, noise=False)

In [ ]:
%%time
backprojected = instrument.backproject(image_separated, integrate=False)

In [ ]:
%%time
scene_degraded = np.mean(backprojected, axis=axis_channel, keepdims=False)
scene_degraded.shape

In [ ]:
scene_degraded = scene_degraded.replace(inputs=scene.inputs)

In [ ]:
backprojected.outputs.sum().to(scene.outputs.unit) / 4

In [ ]:
coords_scene_single = coordinates_scene.replace(wavelength=images.inputs.wavelength)
coords_sensor_single = images.inputs

In [ ]:
images.inputs.wavelength

In [ ]:
instrument_single = dataclasses.replace(
    instrument,
    coordinates_scene=coords_scene_single,
    coordinates_sensor=coords_sensor_single,
)

In [ ]:
s = instrument_single.backproject(images) * scene.outputs.shape[scene.axis_wavelength]

In [ ]:
s.outputs.shape

In [ ]:
smin = np.min(s, axis=axis_channel)[{axis_channel: 0}]

In [ ]:
s.shape

In [ ]:
smin.shape

In [ ]:
scene.shape

In [ ]:
instrument_single.backproject(images).outputs.sum().to(scene.outputs.unit) / 4

In [ ]:
s.outputs.sum().to(scene.outputs.unit) / 4

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots()
    na.plt.pcolormesh(
        smin.inputs.position.x,
        smin.inputs.position.y,
        C=smin.outputs[{axis_channel: 0, scene.axis_wavelength:0 }].value,
        vmin=0,
        vmax=np.percentile(smin.outputs, 99.5).value,
    )
    ax.set_aspect("equal")

In [ ]:
mart = ctis.inverters.MartInverter(
    instrument=instrument,
    intermediate=True,
    num_iteration=20,
)

In [ ]:
spectrum_avg = scene.outputs.mean((scene.axis_detector_x, scene.axis_detector_y))
spectrum_avg = spectrum_avg / spectrum_avg.sum()

In [ ]:
std_OV = esis.flights.f1.spectrum.O_V.width_doppler
v = scene.inputs.velocity.cell_centers(scene.axis_wavelength)
# guess_spectral = np.exp(-np.square(v / std_OV) / 2)
std_OV = 37.5  * 1.5 * u.km / u.s
kappa = 0.2
v0 = 0 * u.km / u.s
guess_spectral = (1 + np.square((v - v0) / std_OV) / kappa)**(-kappa - 1)
guess_spectral = guess_spectral / guess_spectral.sum()

In [ ]:
guess_spatial = smin.outputs
# guess_spatial = guess_spatial.broadcast_to(scene.outputs.shape)
guess_spatial = guess_spatial / guess_spatial.sum()
guess_spatial.sum()

In [ ]:
guess_total = backprojected.outputs.sum().to(scene.outputs.unit) / 4
guess_total

In [ ]:
guess = guess_spectral * guess_spatial * guess_total
# guess = spectrum_avg * scene.outputs.sum()

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.stairs(
        scene.inputs.velocity,
        scene.outputs.mean((scene.axis_detector_x, scene.axis_detector_y)),
        ax=ax,
        label="original",
        axis=scene.axis_wavelength,
    )
    na.plt.stairs(
        scene.inputs.velocity,
        guess.mean((scene.axis_detector_x, scene.axis_detector_y)),
        label="guess",
        axis=scene.axis_wavelength,
    )
    ax.set_xlabel(f"wavelength ({ax.get_xlabel()})")
    ax.set_ylabel(f"average radiance ({ax.get_ylabel()})");
    ax.legend()

In [ ]:
guess.sum()

In [ ]:
scene.outputs.sum()

In [ ]:
%%time
inversion = mart(images, guess=guess, verbose=True)

In [ ]:
axis_iter = inversion.inverter.axis_iteration

In [ ]:
fig, ax = plt.subplots(
    nrows=2,
    sharex=True,
    constrained_layout=True,
)
na.plt.plot(
    inversion.iteration,
    inversion.mean_chi_squared,
    ax=ax[0],
    axis=axis_iter,
    label=channel,
)
na.plt.plot(
    inversion.iteration,
    inversion.correlation_residual,
    ax=ax[1],
    axis=axis_iter,
    label=channel,
)
ax[0].set_ylabel(r"$\langle \chi^2 \rangle$")
ax[1].set_xlabel("iteration")
ax[1].set_ylabel("signal-correlated residual")
ax[0].set_yscale("log")
ax[0].legend();

In [ ]:
solution = inversion.solution

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.stairs(
        scene.inputs.wavelength,
        scene.outputs.mean((scene.axis_detector_x, scene.axis_detector_y)),
        ax=ax,
        label="original",
    )
    na.plt.stairs(
        scene.inputs.wavelength,
        guess.mean((scene.axis_detector_x, scene.axis_detector_y)),
        label="guess",
        axis=scene.axis_wavelength,
    )
    na.plt.stairs(
        solution.inputs.wavelength,
        solution.outputs.mean((scene.axis_detector_x, scene.axis_detector_y)),
        ax=ax,
        label="reconstructed",
    )
    ax.set_xlabel(f"wavelength ({ax.get_xlabel()})")
    # ax2.set_xlabel(f"wavelength ({ax2.get_xlabel()})")
    ax.set_ylabel(f"average radiance ({ax.get_ylabel()})")
    ax.legend()

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=3,
        gridspec_kw=dict(width_ratios=[.5, .5, .1]),
        constrained_layout=True,
        figsize=(10, 6),
    )
    ax1, ax2, cax = axs
    ax2.set_yticklabels([])
    vmin = np.nanpercentile(
        a=scene.outputs,
        q=0.5,
        axis=(scene.axis_detector_x, scene.axis_detector_y),
    )
    vmax = np.nanpercentile(
        a=scene.outputs,
        q=99.5,
        axis=(scene.axis_detector_x, scene.axis_detector_y),
    )
    wavelength_min=-100 * u.km / u.s,
    wavelength_max=+100 * u.km / u.s,
    na.plt.rgbmesh(
        scene.inputs.velocity.cell_centers(scene.axis_wavelength),
        scene.inputs.position.x,
        scene.inputs.position.y,
        C=scene_degraded.outputs,
        axis_wavelength=scene.axis_wavelength,
        ax=ax1,
        vmin=vmin,
        vmax=vmax,
        wavelength_min=-100 * u.km / u.s,
        wavelength_max=+100 * u.km / u.s,
    )
    colorbar = na.plt.rgbmesh(
        scene.inputs.velocity.cell_centers(scene.axis_wavelength),
        scene.inputs.position.x,
        scene.inputs.position.y,
        C=solution.outputs,
        axis_wavelength=scene.axis_wavelength,
        ax=ax2,
        vmin=vmin,
        vmax=vmax,
        wavelength_min=-100 * u.km / u.s,
        wavelength_max=+100 * u.km / u.s,
    )
    na.plt.pcolormesh(
        C=colorbar,
        axis_rgb=scene.axis_wavelength,
        ax=cax,
    )
    ax1.set_title("degraded")
    ax2.set_title("reconstructed")
    unit_x = scene.inputs.position.x.unit
    unit_y = scene.inputs.position.y.unit
    ax1.set_xlabel(f"scene $x$ ({unit_x:latex_inline})")
    ax2.set_xlabel(f"scene $x$ ({unit_x:latex_inline})")
    ax1.set_ylabel(f"scene $y$ ({unit_y:latex_inline})")
    cax.xaxis.set_ticks_position("top")
    cax.xaxis.set_label_position("top")
    cax.yaxis.tick_right()
    cax.yaxis.set_label_position("right")

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(
        ncols=2,
        constrained_layout=True,
        sharex=True,
        sharey=True,
        figsize=(6, 5),
    )
    i = {scene.axis_detector_x: 210}
    vmax = scene_degraded.outputs.percentile(99.9).ndarray.value
    norm = matplotlib.colors.PowerNorm(gamma=0.5, vmin=0, vmax=vmax)
    na.plt.pcolormesh(
        scene.inputs.velocity[{scene.axis_detector_x: 0, scene.axis_detector_y: 0}],
        scene.inputs.position.y[i],
        C=scene_degraded.outputs[i].value,
        ax=ax[0],
        norm=norm,
    )
    ani = na.plt.pcolormovie(
        inversion.iteration,
        scene.inputs.velocity[{scene.axis_detector_x: 0, scene.axis_detector_y: 0}],
        inversion.solutions.inputs.position.y[i],
        C=inversion.solutions.outputs[i].to_value(scene_degraded.outputs.unit),
        ax=ax[1],
        axis_time=inversion.inverter.axis_iteration,
        norm=norm,
    )
    ax[0].set_title("original")
    ax[1].set_title("reconstruction")

result = ani.to_jshtml(fps=10)
result = IPython.display.HTML(result)

ani.save("mart-iris-spectra.gif", dpi=200, fps=30)

plt.close(ani._fig)

result

In [ ]:
inversion.plot_moments(
    truth=scene, 
    axis=scene.axis_wavelength,
    range_radiance=[0, 3e3] * u.erg / u.s / u.sr / u.cm**2,
    range_median=[-60, 60] * u.km / u.s,
    range_iqr=[0, 100] * u.km / u.s,
    percentile_radiance=25,
);

In [ ]:
%%time
predictions = instrument.image(solution, noise=False)

In [ ]:
residual = images - predictions

In [ ]:
fig, ax = na.plt.subplots(
    axis_rows="rows",
    axis_cols="cols",
    nrows=2,
    ncols=2,
    constrained_layout=True,
    figsize=(8,7),
    sharex=True,
    sharey=True,
    origin="upper",
)
ax = ax.reshape(dict(channel=-1))
vmin = -35
vmax = +35
img = na.plt.pcolormesh(
    residual.inputs.position.x,
    residual.inputs.position.y,
    C=residual.outputs.value,
    ax=ax,
    vmin=vmin,
    vmax=vmax,
    cmap="gray",
)
na.plt.set_title
na.plt.set_aspect("equal", ax=ax)
plt.colorbar(
    ax=ax.ndarray,
    mappable=img.ndarray[0],
    label=f"residual ({residual.outputs.unit:latex_inline})",
);
na.plt.set_title(channel, ax=ax);

In [ ]:
float(scipy.stats.spearmanr(predictions.outputs.ndarray.reshape(-1), residual.outputs.ndarray.reshape(-1)).statistic)

In [ ]:
float(scipy.stats.pearsonr(predictions.outputs.ndarray.reshape(-1), residual.outputs.ndarray.reshape(-1)).statistic)

In [ ]:
images_perfect = instrument.image(scene, noise=False)

In [ ]:
inversion.inverter.mean_chi_squared(
    images.outputs,
    images_perfect.outputs,
)

In [ ]:
inversion.inverter.correlation_residual(
    images.outputs,
    predictions.outputs,
).ptp()

In [ ]:
# with astropy.visualization.quantity_support():
#     fig, axs = plt.subplots(
#         ncols=3,
#         gridspec_kw=dict(width_ratios=[.5, .5, .1]),
#         constrained_layout=True,
#         figsize=(10, 6),
#     )
#     ax1, ax2, cax = axs
#     ax2.set_yticklabels([])
#     vmax = np.nanpercentile(
#         a=scene.outputs,
#         q=99.5,
        
#         axis=(scene.axis_detector_x, scene.axis_detector_y),
#     )

#     na.plt.rgbmesh(
#         C=scene_degraded,
#         axis_wavelength="wavelength",
#         ax=ax1,
#         vmin=0,
#         vmax=vmax,
#     )
#     label = "iteration = " + inversion.iteration.to_string_array("%d")
#     chisq_str = r"$\langle \chi^2 \rangle$"
#     label = label + f"\n{chisq_str} = " + inversion.mean_chi_squared.mean("channel").to_string_array("%.03f")
#     ani, colorbar = na.plt.rgbmovie(
#         label,
#         scene.velocity_doppler,
#         scene.inputs.position.x,
#         scene.inputs.position.y,
#         C=inversion.solutions.outputs,
#         axis_time=inversion.inverter.axis_iteration,
#         axis_wavelength="wavelength",
#         ax=ax2,
#         vmin=0,
#         vmax=vmax,
#     )
#     na.plt.pcolormesh(
#         C=colorbar,
#         axis_rgb="wavelength",
#         ax=cax,
#     )
#     ax1.set_title("original")
#     ax2.set_title("reconstructed")
#     unit_x = scene.inputs.position.x.unit
#     unit_y = scene.inputs.position.y.unit
#     ax1.set_xlabel(f"scene $x$ ({unit_x:latex_inline})")
#     ax2.set_xlabel(f"scene $x$ ({unit_x:latex_inline})")
#     ax1.set_ylabel(f"scene $y$ ({unit_y:latex_inline})")
#     cax.xaxis.set_ticks_position("top")
#     cax.xaxis.set_label_position("top")
#     cax.yaxis.tick_right()
#     cax.yaxis.set_label_position("right")

# result = ani.to_jshtml(fps=30)
# result = IPython.display.HTML(result)

# ani.save("mart-iris-rgb.mp4")

# plt.close(ani._fig)

# result

 - mulitply counts by 10 to see how it changes the results
 - smooth the outputs before taking the residual (and computing the moments)
 - negative correlation coefficient implies that we went a little too far (crossed zero)

$d \chi = \frac{d' - d}{\sqrt{d}}$ contribution of the residual to the total chi square (SNR-weighted residual)